# Establishing correctness by comparing outputs

This notebook tests the correctness of the different implementations by comparing outputs to a simple exhaustive loop search or across implmentations.

Note that the notebook can take a couple of minutes of run time.

In [ ]:
n_repeat = 1

## Installation (< 2min)

In [ ]:
try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

compile XTNeighbor-streaming

In [ ]:
! mkdir -p {repo_path}/xtneighbor_streaming/build
! cd {repo_path}/xtneighbor_streaming/build; cmake -S .. -B .;make

compile XTNeighbor

In [ ]:
! mkdir -p {repo_path}/xtneighbor/build
! cd {repo_path}/xtneighbor/build; cmake -S .. -B .;make

## Setup and data import

In [ ]:
from os import path
import numpy as np
import pandas as pd
import seaborn as sns
import time
import subprocess
import re
import random
from pyrepseq import levenshtein_distance
import pyrepseq
import symscan

In [ ]:
import benchutils as bu

bu.describe_env()

read in data

In [ ]:
N_FILES=1

data = []
for i in range(1,N_FILES+1):
  data += pd.read_csv(f'{repo_path}/data/emerson{i}.zip', compression='zip', header=0)['cdr3'].to_list()

print('first row:', data[0]);
print(f'len: {len(data):,}')

Write files functions used for XTNeighbor and XTNeighbor-streaming respectively

In [ ]:
!mkdir -p tmp

In [ ]:
def writeFile(seqs):
  with open("tmp/input.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in seqs)

def writeFile2(seqs):
  with open("tmp/input2.txt","w") as file1:
    file1.writelines(seq+'\n' for seq in (['cdr3']+seqs))

## Implementations

In [ ]:
def xt_neighbor(seqs,threshold,_len): #verbose is ignored
  ! {repo_path}/xtneighbor/build/xt_neighbor -p "tmp/input.txt" -n "$_len" -d "$threshold" -o "tmp/xt_output.txt"
  return read_result('tmp/xt_output.txt')

def xt_neighbor_streaming(seqs,threshold,_len):
  ! {repo_path}/xtneighbor_streaming/build/xt_neighbor -i "tmp/input2.txt" -n "$_len" -d "$threshold" -o "tmp/xt_streaming_output.txt"
  return read_result('tmp/xt_streaming_output.txt')

def symdel(seqs,threshold,_len):
  return set(pyrepseq.symdel(seqs,max_edits=threshold, output_symmetric=False))

def run_symscan(seqs,threshold,_len):
  row, col, dists = symscan.get_neighbors_within(seqs, max_distance=threshold)
  return set(zip(row, col, dists))

def for_loop(seqs,threshold,_len):
  ans = set()
  for i in range(len(seqs)):
    for j in range(len(seqs)):
        if i >= j:
            continue
        dist = levenshtein_distance(seqs[j], seqs[i], score_cutoff=threshold)
        if dist <= threshold:
            ans.add((i, j, dist))
  return ans

def prepare(seqs):
  writeFile(seqs)
  writeFile2(seqs)

def read_result(filename):
  df = pd.read_csv(filename, sep=' ', header=None)
  row_set = set(df.itertuples(index=False, name=None))
  return row_set


## Run on small datasets (< 1 min)

In [ ]:
algorithms = {
    'for_loop':for_loop,
    'symdel':symdel,
    'symscan':run_symscan,
    'xt':xt_neighbor,
    'xt_streaming':xt_neighbor_streaming,

}

def perform(subset, distance, algorithms, size):
    result=None
    for alg_name in algorithms:
        print('running algorithm:', alg_name)
        prepare(subset)
        new_result = algorithms[alg_name](subset,distance,size)
        if result is None:
            result = new_result
        elif result != new_result:
            raise Exception('comparison failed')


def run_exp(distance, size, shuffle=True):
    for i in range(n_repeat):
        subset = random.Random(i).sample(data,size)
        perform(subset, distance, algorithms, size)
    print('success!')

In [ ]:
run_exp(distance=1, size=5000)

In [ ]:
run_exp(distance=2, size=5000)

In [ ]:
run_exp(distance=3, size=5000)

## Run on large datasets (< 5 min)

In [ ]:
algorithms = {
    'xt_streaming':xt_neighbor_streaming,
    'symscan':run_symscan,
}

In [ ]:
run_exp(distance=1, size=3_000_000)

In [ ]:
run_exp(distance=2, size=1_000_000)

In [ ]:
run_exp(distance=3, size=100_000)